<a href="https://colab.research.google.com/github/ekonjmrivas-devops/llm_engineering/blob/mis-ejercicios/week3/W3_PRACTICA_DATOS_SINTETICOS_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 3 - Práctica | Generador de Datos Sintéticos (v5)

---
**Autor:** José María Rivas | Práctica LLM Engineering
**Curso:** UNED Curso LLM Engineering | Práctica Week 3
**Fecha:** 3 de julio de 2026
**Versión:** v5
---

## 📖 Resumen del proyecto

Esta herramienta genera **datasets sintéticos** para cualquier dominio de datos
(no solo el ejemplo de tickets de soporte usado en las pruebas), a partir de tres
piezas independientes que el usuario gestiona desde Google Drive y desde el propio
formulario de Gradio, sin tocar código:

- **Semilla:** un puñado de ejemplos reales que marcan el estilo y la variedad esperada.
- **Esquema:** los campos que debe tener cada registro generado y su tipo. A partir de
  él se construye en tiempo de ejecución tanto el validador (`pydantic.create_model()`)
  como, si el usuario no define uno propio, el prompt de sistema.
- **Prompt de sistema:** las instrucciones de generación, editables y reutilizables.

El motor de generación admite **Claude** (vía tool calling / function calling, que
fuerza la estructura exacta del esquema en la respuesta) o **Llama 3.1 8B en local**
(cuantizado a 4 bits, vía prompting). La generación se hace por lotes, valida cada
registro contra el esquema activo, y guarda el resultado en Drive como JSON o CSV.

La interfaz Gradio organiza todo esto en pestañas (Ayuda, Prompt, Esquema, Semilla,
Generar, Resultados), con una pantalla de arranque que carga el sistema antes de
mostrar el formulario.

**Bloque - Librerías**

In [1]:
!pip install -q anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 6.8 MB/s eta 0:00:00


In [2]:
!pip install -q -U "bitsandbytes>=0.46.1"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00


In [3]:
!pip install -q gradio


In [4]:
import os
import re
import json
import csv
import time
import threading
import subprocess
from datetime import datetime
from google.colab import drive
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import anthropic
from pydantic import ValidationError, create_model
import gradio as gr


**Bloque - Configuración global**

In [5]:
# True: carga Llama de forma eager en el arranque (pantalla de carga), ~12 min.
# False: arranque rápido (solo Claude); Llama seguirá cargándose de forma perezosa
# si más adelante seleccionas "llama" en algún selector del formulario.
CARGAR_LLAMA_AL_INICIO = False


**Bloque - Setup y conexión a Drive**

In [6]:
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
BASE_PATH = '/content/drive/MyDrive/cursollms/week3_sintetico'
SEED_PATH = f'{BASE_PATH}/Seed files'
SCHEMA_PATH = f'{BASE_PATH}/Schema files'
PROMPT_PATH = f'{BASE_PATH}/Prompt files'
OUTPUT_PATH = f'{BASE_PATH}/Outbound files'


In [8]:
for carpeta in (SEED_PATH, SCHEMA_PATH, PROMPT_PATH, OUTPUT_PATH):
    os.makedirs(carpeta, exist_ok=True)


In [9]:
print(f"Semillas: {SEED_PATH}")
print(f"Esquemas: {SCHEMA_PATH}")
print(f"Prompts:  {PROMPT_PATH}")
print(f"Salida:   {OUTPUT_PATH}")


Semillas: /content/drive/MyDrive/cursollms/week3_sintetico/Seed files
Esquemas: /content/drive/MyDrive/cursollms/week3_sintetico/Schema files
Prompts:  /content/drive/MyDrive/cursollms/week3_sintetico/Prompt files
Salida:   /content/drive/MyDrive/cursollms/week3_sintetico/Outbound files


**Bloque - Función genérica para eliminar archivos**

In [10]:
def eliminar_archivo(ruta_base, nombre_archivo):
    """
    Elimina físicamente un fichero de una de las carpetas de Drive gestionadas
    por la herramienta (Seed files, Schema files, Prompt files, Outbound files).
    Devuelve True si se eliminó, False si no existía. Acción irreversible.
    """
    ruta = os.path.join(ruta_base, nombre_archivo)
    if os.path.exists(ruta):
        os.remove(ruta)
        print(f"🗑️ Archivo eliminado: {ruta}")
        return True
    return False


**Bloque - Listar archivos de semilla disponibles en Drive**

In [11]:
def listar_archivos_semilla():
    """Lista los archivos JSON disponibles en Seed files, para el Dropdown de Gradio."""
    archivos = [f for f in os.listdir(SEED_PATH) if f.lower().endswith(".json")]
    archivos.sort()
    return archivos


**Bloque - Carga y guardado del dataset semilla en Drive**

In [12]:
def cargar_dataset_semilla(nombre_archivo):
    """Carga el dataset semilla desde un archivo JSON depositado en Seed files."""
    ruta = os.path.join(SEED_PATH, nombre_archivo)
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró '{nombre_archivo}' en {SEED_PATH}")
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)


In [13]:
def guardar_dataset_semilla(datos, nombre_archivo):
    """Guarda una lista de registros semilla como JSON en Seed files."""
    if not nombre_archivo.lower().endswith(".json"):
        nombre_archivo += ".json"
    ruta = os.path.join(SEED_PATH, nombre_archivo)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)
    print(f"✅ Semilla guardada en: {ruta}")
    return ruta


**Bloque - Listar, cargar y guardar esquemas de validación en Drive**

In [14]:
def listar_archivos_esquema():
    """Lista los archivos JSON disponibles en Schema files."""
    archivos = [f for f in os.listdir(SCHEMA_PATH) if f.lower().endswith(".json")]
    archivos.sort()
    return archivos


In [15]:
def cargar_esquema(nombre_archivo):
    """
    Carga un esquema de validación desde Schema files.
    Formato esperado: {"campos": [{"nombre":..., "tipo":...}, ...], "campo_id": "nombre_o_null"}
    """
    ruta = os.path.join(SCHEMA_PATH, nombre_archivo)
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró '{nombre_archivo}' en {SCHEMA_PATH}")
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)


In [16]:
def guardar_esquema(esquema, nombre_archivo):
    """Guarda un esquema de validación como JSON en Schema files."""
    if not nombre_archivo.lower().endswith(".json"):
        nombre_archivo += ".json"
    ruta = os.path.join(SCHEMA_PATH, nombre_archivo)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(esquema, f, ensure_ascii=False, indent=2)
    print(f"✅ Esquema guardado en: {ruta}")
    return ruta


**Bloque - Construcción dinámica del modelo de validación (Pydantic)**

In [17]:
TIPO_MAP = {"str": str, "int": int, "float": float, "bool": bool}

def construir_modelo_validacion(esquema):
    """
    Construye una clase Pydantic en tiempo de ejecución a partir de un esquema
    genérico (lista de campos con nombre y tipo).
    """
    campos_modelo = {}
    for campo in esquema["campos"]:
        tipo_python = TIPO_MAP.get(campo["tipo"], str)
        campos_modelo[campo["nombre"]] = (tipo_python, ...)
    return create_model("ModeloDinamico", **campos_modelo)


**Bloque - Listar, cargar y guardar prompts de sistema en Drive**

In [18]:
def listar_archivos_prompt():
    """Lista los archivos JSON disponibles en Prompt files."""
    archivos = [f for f in os.listdir(PROMPT_PATH) if f.lower().endswith(".json")]
    archivos.sort()
    return archivos


In [19]:
def cargar_prompt(nombre_archivo):
    """Carga un prompt de sistema guardado (formato {"contenido": texto})."""
    ruta = os.path.join(PROMPT_PATH, nombre_archivo)
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró '{nombre_archivo}' en {PROMPT_PATH}")
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)["contenido"]


In [20]:
def guardar_prompt(texto, nombre_archivo):
    """Guarda un prompt de sistema como JSON en Prompt files."""
    if not nombre_archivo.lower().endswith(".json"):
        nombre_archivo += ".json"
    ruta = os.path.join(PROMPT_PATH, nombre_archivo)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump({"contenido": texto}, f, ensure_ascii=False, indent=2)
    print(f"✅ Prompt guardado en: {ruta}")
    return ruta


**Bloque - Prompt de sistema automático a partir del esquema**

In [21]:
def construir_prompt_sistema_automatico(esquema):
    """
    Genera un system prompt genérico a partir de los campos del esquema.
    Se usa como valor por defecto cuando el usuario no ha seleccionado
    ni escrito un prompt propio.
    """
    campos_txt = ", ".join(c["nombre"] for c in esquema["campos"])

    prompt = f"""Eres un generador de datos sintéticos. Tu tarea es crear registros NUEVOS
y REALISTAS basados en el estilo, formato y variedad de los ejemplos semilla proporcionados.

Reglas estrictas:
1. Responde ÚNICAMENTE con un array JSON válido, sin texto adicional ni explicaciones.
2. Cada registro debe tener exactamente estos campos: {campos_txt}.
3. NO repitas literalmente los ejemplos; genera situaciones distintas pero plausibles dentro del mismo dominio.
4. Mantén coherencia de estilo, longitud y nivel de detalle con los ejemplos semilla."""

    campo_id = esquema.get("campo_id")
    if campo_id:
        prompt += f"\n5. El campo '{campo_id}' debe continuar la numeración/formato del último ejemplo semilla."

    return prompt


**Bloque - Refuerzo del prompt con los campos del esquema activo**

In [22]:
def reforzar_prompt_con_esquema(system_prompt, esquema):
    """
    Añade al final del system prompt (sea el guardado o el auto-generado) un
    recordatorio no negociable con los campos exactos del esquema activo.
    Evita que un prompt guardado/editado por separado quede desincronizado
    si el esquema cambia después.
    """
    campos_txt = ", ".join(c["nombre"] for c in esquema["campos"])
    return system_prompt + f"""

IMPORTANTE - ESTO PREVALECE SOBRE CUALQUIER INSTRUCCIÓN ANTERIOR:
Los campos obligatorios y exactos de cada registro son: {campos_txt}.
No omitas ninguno de estos campos aunque una instrucción previa mencione una lista distinta."""


**Bloque - Cálculo genérico del siguiente ID**

In [23]:
def calcular_siguiente_id(seed_dataset, campo_id):
    """
    Calcula el siguiente valor de ID a partir del último ejemplo semilla,
    detectando el número final de la cadena (ej. 'TCK-1001' -> 'TCK-1002').
    Devuelve None si el esquema no define campo_id o no se detecta un patrón numérico.
    """
    if not campo_id or not seed_dataset:
        return None

    ultimo_valor = str(seed_dataset[-1].get(campo_id, ""))
    match = re.search(r"(\d+)$", ultimo_valor)
    if not match:
        return None

    numero = match.group(1)
    prefijo = ultimo_valor[:match.start()]
    siguiente_numero = str(int(numero) + 1).zfill(len(numero))
    return f"{prefijo}{siguiente_numero}"


**Bloque - Construcción del prompt de usuario**

In [24]:
def construir_prompt_usuario(seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales=""):
    """
    Construye el prompt de usuario con los ejemplos semilla (few-shot),
    la petición de generación y, opcionalmente, instrucciones adicionales.
    """
    prompt = f"""Aquí tienes ejemplos semilla de referencia:

{json.dumps(seed_dataset, indent=2, ensure_ascii=False)}

Genera {num_ejemplos} registros NUEVOS siguiendo el mismo estilo y estructura."""

    if siguiente_id:
        prompt += f"\nEmpieza la numeración en: {siguiente_id}"

    if instrucciones_adicionales and instrucciones_adicionales.strip():
        prompt += f"\n\nInstrucciones adicionales:\n{instrucciones_adicionales.strip()}"

    prompt += "\n\nResponde solo con el array JSON."
    return prompt


**Bloque - Configuración de cuantización y carga de Llama local**

In [25]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)


In [26]:
# Carga el modelo de Llama solo cuando sea necesario (idempotente)

LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

llama_tokenizer = None
llama_model = None

def cargar_llama_si_necesario():
    """
    Carga el tokenizer y el modelo Llama en memoria si aún no están cargados.
    Si ya se cargaron en esta sesión, la llamada es prácticamente instantánea.
    Comprueba que hay GPU disponible antes de intentar cargar, para dar un
    error claro en vez de un fallo críptico de CUDA.
    """
    global llama_tokenizer, llama_model
    if llama_model is None:
        if not torch.cuda.is_available():
            raise RuntimeError(
                "No se ha detectado GPU en este entorno. El modelo Llama requiere GPU "
                "(en Colab: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4)."
            )
        print("⏳ Cargando modelo Llama local, puede tardar varios minutos la primera vez...")
        llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA)
        llama_model = AutoModelForCausalLM.from_pretrained(
            LLAMA,
            device_map="auto",
            quantization_config=quant_config
        )
        print("✅ Modelo Llama cargado.")


**Bloque - Configuración Claude**

In [27]:
CLAUDE = "claude-sonnet-4-6"

claude_client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)


**Bloque - Definición de la herramienta (tool) para forzar el esquema**

In [28]:
JSON_SCHEMA_TIPO_MAP = {"str": "string", "int": "integer", "float": "number", "bool": "boolean"}

def construir_tool_generacion(esquema):
    """
    Construye la definición de herramienta (tool) de Claude a partir del esquema activo.
    En vez de pedir "responde solo con JSON" en texto libre (que el modelo puede
    interpretar mal), se define un input_schema exacto y se fuerza su uso —
    la respuesta queda estructuralmente obligada a cumplir el esquema.
    """
    propiedades = {}
    for campo in esquema["campos"]:
        tipo_json = JSON_SCHEMA_TIPO_MAP.get(campo["tipo"], "string")
        propiedades[campo["nombre"]] = {"type": tipo_json}

    return {
        "name": "generar_registros",
        "description": "Genera una lista de registros sintéticos que cumplen exactamente el esquema indicado.",
        "input_schema": {
            "type": "object",
            "properties": {
                "registros": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": propiedades,
                        "required": list(propiedades.keys())
                    }
                }
            },
            "required": ["registros"]
        }
    }


**Bloque - Generación con Claude**

In [29]:
def generar_con_claude(system_prompt, seed_dataset, num_ejemplos, siguiente_id, esquema, instrucciones_adicionales=""):
    """
    Genera registros sintéticos usando Claude API con tool calling (function calling):
    se define una herramienta cuyo input_schema coincide con el esquema activo y se
    fuerza su uso (tool_choice), en vez de depender de que el modelo "interprete"
    una instrucción de formato en texto libre.

    Devuelve el resultado como texto JSON (mismo formato que la versión anterior basada
    en prompting), para que el resto del pipeline (parsear_y_validar, diagnóstico) no
    tenga que cambiar.
    """
    user_prompt = construir_prompt_usuario(seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales)
    tool = construir_tool_generacion(esquema)

    response = claude_client.messages.create(
        model=CLAUDE,
        max_tokens=4000,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
        tools=[tool],
        tool_choice={"type": "tool", "name": "generar_registros"}
    )

    for bloque in response.content:
        if bloque.type == "tool_use" and bloque.name == "generar_registros":
            registros = bloque.input.get("registros", [])
            return json.dumps(registros, ensure_ascii=False)

    return "[]"


**Bloque - Generación con Llama**

In [30]:
def generar_con_llama(system_prompt, seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales=""):
    """
    Genera registros sintéticos usando Llama local. Devuelve el texto crudo de la respuesta.
    A diferencia de Claude, aquí no se usa tool calling: Llama 3.1 en local no tiene un
    soporte de function calling tan directo como la API de Claude, así que sigue
    dependiendo de la instrucción de formato en el prompt + validación posterior con Pydantic.
    """
    cargar_llama_si_necesario()

    user_prompt = construir_prompt_usuario(seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales)

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_prompt}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_prompt}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

    inputs = llama_tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=4000
    )

    return llama_tokenizer.decode(outputs[0], skip_special_tokens=True)


**Bloque - Función orquestadora del modelo**

In [31]:
def generar_registros_raw(system_prompt, seed_dataset, num_ejemplos, siguiente_id, esquema, modelo="claude", instrucciones_adicionales=""):
    """
    Genera registros con el modelo seleccionado. modelo: "claude" o "llama".
    esquema se usa solo en la rama Claude, para construir la herramienta (tool calling)
    que fuerza la estructura exacta de la respuesta.
    """
    if modelo == "claude":
        return generar_con_claude(system_prompt, seed_dataset, num_ejemplos, siguiente_id, esquema, instrucciones_adicionales)
    elif modelo == "llama":
        return generar_con_llama(system_prompt, seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales)
    else:
        raise ValueError(f"Modelo no reconocido: {modelo}")


**Bloque - Parseo y validación de la salida**

In [32]:
def parsear_y_validar(texto_generado, modelo_validacion):
    """
    Limpia la salida del modelo, la parsea como JSON y valida cada registro
    contra el modelo Pydantic dinámico. Devuelve (registros_validos, errores).
    """
    limpio = texto_generado.replace("```json", "").replace("```", "").strip()

    try:
        datos = json.loads(limpio)
    except json.JSONDecodeError as e:
        return [], [f"Error de parseo JSON: {e}"]

    registros_validos = []
    errores = []

    for item in datos:
        try:
            registro = modelo_validacion(**item)
            registros_validos.append(registro.model_dump())
        except ValidationError as e:
            errores.append(f"Registro inválido ({item}): {e}")

    return registros_validos, errores


**Bloque - Generación por lotes**

In [33]:
TAMANO_LOTE = 20  # nº máximo de registros por llamada al modelo

def generar_dataset_por_lotes(seed_dataset, esquema, system_prompt, num_total, modelo="claude", instrucciones_adicionales=""):
    """
    Genera el número total de registros solicitado, dividiendo la petición en lotes.
    El prompt se refuerza una vez con los campos del esquema activo antes de empezar.
    Devuelve (registros_generados, errores_acumulados).
    """
    modelo_validacion = construir_modelo_validacion(esquema)
    campo_id = esquema.get("campo_id")
    system_prompt_reforzado = reforzar_prompt_con_esquema(system_prompt, esquema)

    contexto_seed = list(seed_dataset)
    registros_generados = []
    errores_acumulados = []
    pendientes = num_total

    while pendientes > 0:
        lote = min(TAMANO_LOTE, pendientes)
        siguiente_id = calcular_siguiente_id(contexto_seed, campo_id)

        print(f"⏳ Generando lote de {lote} registros...")
        texto_crudo = generar_registros_raw(
            system_prompt_reforzado, contexto_seed, lote, siguiente_id, esquema,
            modelo=modelo, instrucciones_adicionales=instrucciones_adicionales
        )
        validos, errores = parsear_y_validar(texto_crudo, modelo_validacion)

        registros_generados.extend(validos)
        errores_acumulados.extend(errores)
        if validos and campo_id:
            contexto_seed = contexto_seed + validos
        pendientes -= lote

        print(f"✅ Lote completado: {len(validos)} válidos, {len(errores)} con error")

    return registros_generados, errores_acumulados


**Bloque - Función auxiliar de llamada simple a Llama**

In [34]:
def llamar_llama_simple(instruccion, max_tokens=2000):
    """
    Llamada simple a Llama local con una única instrucción (sin roles
    system/user separados). La usan las funciones auxiliares de IA
    (prompt/esquema/semilla) cuando se elige Llama como motor.
    Decodifica solo los tokens generados, no el prompt de entrada.
    """
    cargar_llama_si_necesario()

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{instruccion}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

    inputs = llama_tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = llama_model.generate(**inputs, max_new_tokens=max_tokens)

    tokens_generados = outputs[0][inputs["input_ids"].shape[1]:]
    return llama_tokenizer.decode(tokens_generados, skip_special_tokens=True).strip()


**Bloque - Funciones auxiliares con IA (enriquecer prompt, validar esquema, formatear semilla)**

In [35]:
def enriquecer_prompt_con_ia(texto_borrador, modelo="claude"):
    """
    Envía un borrador de system prompt a la IA seleccionada para enriquecerlo:
    más claro, más completo, sin perder la intención original.
    """
    instruccion = f"""Eres un experto en prompt engineering. Mejora el siguiente system prompt
para un generador de datos sintéticos: hazlo más claro, completo y con reglas explícitas de formato,
sin cambiar su intención original. Responde ÚNICAMENTE con el prompt mejorado, sin explicaciones.

Prompt original:
{texto_borrador}
"""
    if modelo == "claude":
        response = claude_client.messages.create(
            model=CLAUDE,
            max_tokens=1500,
            messages=[{"role": "user", "content": instruccion}]
        )
        return response.content[0].text.strip()
    elif modelo == "llama":
        return llamar_llama_simple(instruccion, max_tokens=1500)
    else:
        raise ValueError(f"Modelo no reconocido: {modelo}")


In [36]:
def validar_formatear_esquema_con_ia(texto_borrador, modelo="claude"):
    """
    Envía una descripción libre de campos a la IA seleccionada y devuelve un esquema
    JSON válido con el formato {"campos": [{"nombre":..., "tipo":...}], "campo_id": ...}.
    Los nombres de campo se normalizan a snake_case para evitar problemas de
    coincidencia de claves JSON con espacios o mayúsculas.
    """
    instruccion = f"""Convierte la siguiente descripción de campos en un esquema JSON válido
con este formato exacto:
{{"campos": [{{"nombre": "...", "tipo": "str|int|float|bool"}}, ...], "campo_id": "nombre_del_campo_id_o_null"}}

Los nombres de los campos deben normalizarse a snake_case (minúsculas, sin espacios ni acentos,
palabras separadas por guion bajo). Por ejemplo, "Nivel de Impacto" debe quedar como "nivel_impacto".

Responde ÚNICAMENTE con el JSON, sin texto adicional.

Descripción:
{texto_borrador}
"""
    if modelo == "claude":
        response = claude_client.messages.create(
            model=CLAUDE,
            max_tokens=1000,
            messages=[{"role": "user", "content": instruccion}]
        )
        texto_respuesta = response.content[0].text
    elif modelo == "llama":
        texto_respuesta = llamar_llama_simple(instruccion, max_tokens=1000)
    else:
        raise ValueError(f"Modelo no reconocido: {modelo}")

    limpio = texto_respuesta.replace("```json", "").replace("```", "").strip()
    return json.loads(limpio)


In [37]:
def formatear_semilla_con_ia(texto_borrador, esquema=None, modelo="claude"):
    """
    Envía datos semilla pegados en bruto (texto libre, CSV, etc.) a la IA seleccionada
    y devuelve una lista de registros en JSON, ajustada al esquema si se indica.
    """
    contexto_esquema = ""
    if esquema:
        campos_txt = ", ".join(c["nombre"] for c in esquema["campos"])
        contexto_esquema = f"\nLos registros deben tener exactamente estos campos: {campos_txt}."

    instruccion = f"""Convierte los siguientes datos en una lista JSON de registros bien formada.
{contexto_esquema}
Responde ÚNICAMENTE con el array JSON, sin texto adicional.

Datos:
{texto_borrador}
"""
    if modelo == "claude":
        response = claude_client.messages.create(
            model=CLAUDE,
            max_tokens=2000,
            messages=[{"role": "user", "content": instruccion}]
        )
        texto_respuesta = response.content[0].text
    elif modelo == "llama":
        texto_respuesta = llamar_llama_simple(instruccion, max_tokens=2000)
    else:
        raise ValueError(f"Modelo no reconocido: {modelo}")

    limpio = texto_respuesta.replace("```json", "").replace("```", "").strip()
    return json.loads(limpio)


**Bloque - Listar y cargar archivos de resultados generados (Outbound files)**

In [38]:
def listar_archivos_salida():
    """Lista los archivos JSON y CSV disponibles en Outbound files, para el Dropdown de Gradio."""
    archivos = [f for f in os.listdir(OUTPUT_PATH) if f.lower().endswith((".json", ".csv"))]
    archivos.sort()
    return archivos


In [39]:
def cargar_archivo_salida(nombre_archivo):
    """
    Carga y devuelve el contenido de un fichero de resultados ya generado.
    JSON se muestra formateado; CSV se muestra tal cual.
    """
    ruta = os.path.join(OUTPUT_PATH, nombre_archivo)
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró '{nombre_archivo}' en {OUTPUT_PATH}")

    if nombre_archivo.lower().endswith(".json"):
        with open(ruta, "r", encoding="utf-8") as f:
            datos = json.load(f)
        return json.dumps(datos, indent=2, ensure_ascii=False)
    else:
        with open(ruta, "r", encoding="utf-8") as f:
            return f.read()


**Bloque - Generación automática del nombre del archivo**

In [40]:
def generar_nombre_archivo(prefijo="dataset_sintetico", extension="json"):
    """Genera un nombre de archivo de salida basado en un prefijo y timestamp."""
    ahora = datetime.now()
    timestamp = ahora.strftime("%Y%m%d_%H%M%S")
    return f"{prefijo}_{timestamp}.{extension}"


**Bloque - Guardar resultado en Google Drive**

In [41]:
def guardar_json(registros, nombre_archivo):
    """Guarda la lista de registros en un archivo JSON en la carpeta de salida."""
    ruta_completa = os.path.join(OUTPUT_PATH, nombre_archivo)
    with open(ruta_completa, "w", encoding="utf-8") as f:
        json.dump(registros, f, ensure_ascii=False, indent=2)
    print(f"✅ Archivo JSON guardado en: {ruta_completa}")
    return ruta_completa


In [42]:
def guardar_csv(registros, nombre_archivo):
    """Guarda la lista de registros en un archivo CSV en la carpeta de salida."""
    ruta_completa = os.path.join(OUTPUT_PATH, nombre_archivo)
    campos = list(registros[0].keys()) if registros else []
    with open(ruta_completa, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        writer.writeheader()
        writer.writerows(registros)
    print(f"✅ Archivo CSV guardado en: {ruta_completa}")
    return ruta_completa


**Bloque - Función orquestadora para guardar archivo**

In [43]:
def guardar_resultado(registros, nombre_archivo, formato="json"):
    """Guarda el resultado en el formato indicado. formato: "json" o "csv" """
    if formato == "json":
        return guardar_json(registros, nombre_archivo)
    elif formato == "csv":
        return guardar_csv(registros, nombre_archivo)
    else:
        raise ValueError(f"Formato no reconocido: {formato}")


**Bloque - Función orquestadora principal**

In [44]:
def generar_dataset_sintetico(num_ejemplos, modelo, formato_salida, nombre_semilla, esquema, system_prompt,
                                nombre_salida=None, instrucciones_adicionales=""):
    """
    Orquesta el flujo completo de generación de un dataset sintético.
    Requiere nombre_semilla, esquema y system_prompt ya resueltos.
    Devuelve: (registros, ruta_archivo_guardado, errores)
    """
    if not nombre_semilla or not esquema or not system_prompt:
        raise ValueError("Faltan parámetros obligatorios: semilla, esquema y prompt de sistema.")

    print(f"🔧 Ejemplos solicitados: {num_ejemplos} | Modelo: {modelo} | Formato: {formato_salida}")
    print("─" * 50)

    print("⏳ Paso 1: Cargando dataset semilla...")
    seed_dataset = cargar_dataset_semilla(nombre_semilla)
    print(f"✅ Semilla cargada ({len(seed_dataset)} ejemplos)")

    print(f"⏳ Paso 2: Generando registros con {modelo}...")
    registros, errores = generar_dataset_por_lotes(
        seed_dataset, esquema, system_prompt, num_ejemplos,
        modelo=modelo, instrucciones_adicionales=instrucciones_adicionales
    )
    print(f"✅ Generación completada: {len(registros)} registros válidos, {len(errores)} errores")

    if nombre_salida is None or nombre_salida.strip() == "":
        nombre_salida = generar_nombre_archivo(extension=formato_salida)
    else:
        nombre_salida = f"{os.path.splitext(nombre_salida)[0]}.{formato_salida}"

    print(f"⏳ Paso 3: Guardando resultado como {nombre_salida}...")
    ruta = guardar_resultado(registros, nombre_salida, formato=formato_salida)
    print(f"✅ Archivo guardado en: {ruta}")
    print("─" * 50)
    print("🎉 Proceso completado")

    return registros, ruta, errores


**Bloque - Estado de recursos (GPU)**

In [45]:
def obtener_estado_recursos():
    """
    Devuelve un resumen compacto del estado de la GPU (memoria usada/total, % uso),
    equivalente a !nvidia-smi pero capturable como texto dentro de una función.
    """
    try:
        resultado = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
             "--format=csv,noheader"],
            capture_output=True, text=True, check=True
        )
        return resultado.stdout.strip()
    except Exception as e:
        return f"No se pudo obtener el estado de la GPU: {e}"


**Bloque - Función de arranque del sistema (pantalla de carga)**

In [46]:
def cargar_sistema():
    """
    Función generadora que ejecuta la carga inicial del sistema y va
    emitiendo mensajes de estado para la pantalla de carga de Gradio.
    Al terminar, revela el botón "Continuar" en vez de pasar automáticamente
    al formulario principal, para que el log final no desaparezca de golpe.

    La carga eager de Llama se controla con la variable global
    CARGAR_LLAMA_AL_INICIO (ver Bloque - Configuración global). Si está en
    False, el arranque es prácticamente instantáneo y Llama solo se carga
    más adelante, de forma perezosa, si se selecciona en el formulario.
    """
    log = "⏳ Comprobando conexión a Drive...\n"
    yield log, gr.update(visible=False)

    log += "✅ Drive conectado.\n⏳ Configurando cliente Claude...\n"
    yield log, gr.update(visible=False)

    if not CARGAR_LLAMA_AL_INICIO:
        log += "✅ Cliente Claude listo.\n⏭️ Carga de Llama desactivada (CARGAR_LLAMA_AL_INICIO=False).\n"
        log += "🎉 Sistema listo. Pulsa \'Continuar\' para acceder al formulario.\n"
        yield log, gr.update(visible=True)
        return

    log += "✅ Cliente Claude listo.\n⏳ Cargando modelo Llama local (puede tardar ~12 min)...\n"
    yield log, gr.update(visible=False)

    log += f"📊 GPU antes de cargar Llama:\n{obtener_estado_recursos()}\n"
    yield log, gr.update(visible=False)

    resultado = {}
    def _cargar():
        cargar_llama_si_necesario()
        resultado["listo"] = True

    hilo = threading.Thread(target=_cargar)
    hilo.start()

    inicio = time.time()
    while hilo.is_alive():
        time.sleep(5)
        transcurrido = int(time.time() - inicio)
        yield log + f"   ...cargando ({transcurrido // 60} min {transcurrido % 60} s transcurridos)", gr.update(visible=False)

    log += f"✅ Modelo Llama cargado ({int(time.time() - inicio)} s).\n"
    log += f"📊 GPU tras cargar Llama:\n{obtener_estado_recursos()}\n"
    log += "🎉 Sistema listo. Pulsa \'Continuar\' para acceder al formulario.\n"
    yield log, gr.update(visible=True)


**Bloque - Contenido de la pestaña de Ayuda**

In [47]:
TEXTO_AYUDA = """
## Cómo usar este generador

**¿Qué hace cada elemento?**
- 🌱 **Semilla:** ejemplos reales (2-5 suele bastar) que sirven de referencia de estilo,
  formato y nivel de detalle. El modelo genera registros NUEVOS inspirados en estos,
  no los repite.
- 🧩 **Esquema:** define qué campos debe tener cada registro generado y de qué tipo
  (texto, número...). Se usa para validar automáticamente que la IA no se salte ni
  invente campos.
- 📝 **Prompt de sistema:** las instrucciones que guían el estilo y las reglas de
  generación (por ejemplo, qué categorías usar, cómo numerar los IDs). Si no seleccionas
  ninguno, se construye uno automático a partir del esquema.

**Flujo recomendado:**
1. Ve a la pestaña 🌱 Dataset semilla: selecciona un fichero existente y pulsa "✅ Usar",
   o pega datos nuevos, dale formato con IA y pulsa "💾 Guardar y usar"
2. Repite el mismo proceso en 🧩 Esquema de validación
3. Repite el mismo proceso en 📝 Prompt de sistema (opcional)
4. Ve a 🚀 Generar dataset: comprueba que los 3 ficheros activos son los correctos,
   configura nº de ejemplos/modelo/formato y pulsa "Generar dataset"
5. Consulta o elimina resultados ya generados en 📁 Resultados generados

**Diferencia entre botones:**
- "✅ Usar" activa un fichero ya existente tal cual está en Drive, sin modificarlo
- "💾 Guardar y usar" guarda el contenido del cuadro de texto (nuevo o editado) y lo activa
- "🗑️ Borrar" limpia los campos de esa pestaña, sin afectar a lo ya activado
- "🗑️ Eliminar archivo de Drive" borra el fichero físicamente — acción irreversible

**Nota sobre el modelo Llama:** requiere GPU y puede tardar varios minutos en cargar
la primera vez. Si solo vas a usar Claude, no es necesario esperar a que termine.

**Nota sobre fiabilidad (Claude vs Llama):** con Claude, la estructura del registro
generado se fuerza mediante tool calling (function calling) — el modelo queda obligado
a devolver el esquema exacto. Con Llama se sigue dependiendo de que el modelo interprete
bien la instrucción de formato, por lo que es más probable ver errores de validación.
"""


**Bloque - Interfaz Gradio (pantalla de carga + pestañas de gestión)**

In [48]:
with gr.Blocks(title="Generador de Datos Sintéticos") as app:

    # Estado con los ficheros/valores activos para la generación
    semilla_activa = gr.State(None)     # nombre de archivo en Seed files
    esquema_activo = gr.State(None)     # dict del esquema cargado
    prompt_activo = gr.State(None)      # texto del system prompt resuelto

    # ── Pantalla de carga ──────────────────────────────────────
    with gr.Group(visible=True) as grupo_carga:
        gr.Markdown("# 🧪 Generador de Datos Sintéticos")
        gr.Markdown("Inicializando sistema, por favor espera...")
        log_carga = gr.Textbox(label="Registro de arranque", lines=12, interactive=False)
        btn_continuar = gr.Button("➡️ Continuar al formulario", visible=False, variant="primary")

    # ── Formulario principal ───────────────────────────────────
    with gr.Group(visible=False) as grupo_principal:
        with gr.Tabs():

            # --- Pestaña: Ayuda ---
            with gr.TabItem("❓ Ayuda"):
                gr.Markdown(TEXTO_AYUDA)

            # --- Pestaña: Prompt de sistema ---
            with gr.TabItem("📝 Prompt de sistema"):
                gr.Markdown("Selecciona un prompt existente o escribe uno nuevo.")
                prompt_dropdown = gr.Dropdown(choices=listar_archivos_prompt(), label="Prompts guardados")
                btn_refrescar_prompt = gr.Button("🔄 Refrescar", size="sm")
                prompt_texto = gr.Textbox(label="Contenido del prompt", lines=10)
                with gr.Row():
                    btn_cargar_prompt = gr.Button("📂 Cargar seleccionado")
                    btn_usar_prompt = gr.Button("✅ Usar")
                motor_ia_prompt = gr.Radio(choices=["claude", "llama"], value="claude", label="Motor IA")
                gr.Markdown("*⚠️ Llama es menos fiable que Claude para esta tarea de enriquecimiento de texto.*")
                btn_mejorar_prompt = gr.Button("✨ Mejorar con IA")
                prompt_nombre_guardar = gr.Textbox(label="Nombre para guardar (opcional)")
                with gr.Row():
                    btn_guardar_prompt = gr.Button("💾 Guardar y usar", variant="primary")
                    btn_borrar_prompt = gr.Button("🗑️ Borrar", variant="stop")
                    btn_eliminar_prompt = gr.Button("🗑️ Eliminar archivo de Drive", variant="stop")
                prompt_estado = gr.Textbox(label="Estado", interactive=False)

            # --- Pestaña: Esquema de validación ---
            with gr.TabItem("🧩 Esquema de validación"):
                gr.Markdown("Selecciona un esquema existente o describe los campos y valida con IA.")
                esquema_dropdown = gr.Dropdown(choices=listar_archivos_esquema(), label="Esquemas guardados")
                btn_refrescar_esquema = gr.Button("🔄 Refrescar", size="sm")
                esquema_texto = gr.Textbox(
                    label="Descripción de campos (o JSON ya formado)",
                    lines=8,
                    placeholder="Ej: id (texto), categoria (texto), prioridad (texto), importe (numero)..."
                )
                with gr.Row():
                    btn_cargar_esquema = gr.Button("📂 Cargar seleccionado")
                    btn_usar_esquema = gr.Button("✅ Usar")
                motor_ia_esquema = gr.Radio(choices=["claude", "llama"], value="claude", label="Motor IA")
                gr.Markdown("*⚠️ Llama es menos fiable que Claude generando JSON estructurado para esquemas.*")
                btn_validar_esquema = gr.Button("✨ Validar/formatear con IA")
                esquema_nombre_guardar = gr.Textbox(label="Nombre para guardar (opcional)")
                with gr.Row():
                    btn_guardar_esquema = gr.Button("💾 Guardar y usar", variant="primary")
                    btn_borrar_esquema = gr.Button("🗑️ Borrar", variant="stop")
                    btn_eliminar_esquema = gr.Button("🗑️ Eliminar archivo de Drive", variant="stop")
                esquema_estado = gr.Textbox(label="Estado", interactive=False)

            # --- Pestaña: Dataset semilla ---
            with gr.TabItem("🌱 Dataset semilla"):
                gr.Markdown("Selecciona una semilla existente o pega datos en bruto y da formato con IA.")
                semilla_dropdown = gr.Dropdown(choices=listar_archivos_semilla(), label="Semillas guardadas")
                btn_refrescar_semilla = gr.Button("🔄 Refrescar", size="sm")
                semilla_texto = gr.Textbox(label="Datos semilla (pegar en bruto o JSON)", lines=10)
                with gr.Row():
                    btn_cargar_semilla = gr.Button("📂 Cargar seleccionada")
                    btn_usar_semilla = gr.Button("✅ Usar")
                motor_ia_semilla = gr.Radio(choices=["claude", "llama"], value="claude", label="Motor IA")
                gr.Markdown("*⚠️ Llama es menos fiable que Claude generando JSON estructurado para semillas.*")
                btn_formatear_semilla = gr.Button("✨ Formatear con IA")
                semilla_nombre_guardar = gr.Textbox(label="Nombre para guardar (opcional)")
                with gr.Row():
                    btn_guardar_semilla = gr.Button("💾 Guardar y usar", variant="primary")
                    btn_borrar_semilla = gr.Button("🗑️ Borrar", variant="stop")
                    btn_eliminar_semilla = gr.Button("🗑️ Eliminar archivo de Drive", variant="stop")
                semilla_estado = gr.Textbox(label="Estado", interactive=False)

            # --- Pestaña: Generar dataset ---
            with gr.TabItem("🚀 Generar dataset"):
                gr.Markdown("### Ficheros activos")
                with gr.Row():
                    semilla_activa_display = gr.Textbox(label="Semilla", interactive=False)
                    esquema_activo_display = gr.Textbox(label="Esquema", interactive=False)
                    prompt_activo_display = gr.Textbox(label="Prompt de sistema", interactive=False)
                esquema_campos_display = gr.Textbox(label="Campos del esquema activo", interactive=False)

                gr.Markdown("### Configuración")
                num_ejemplos = gr.Number(value=10, label="Nº de ejemplos a generar", precision=0)
                modelo = gr.Radio(choices=["claude", "llama"], value="claude", label="Modelo de IA")
                formato_salida = gr.Radio(choices=["json", "csv"], value="json", label="Formato de salida")
                instrucciones_adicionales = gr.Textbox(
                    label="Instrucciones adicionales (opcional)",
                    placeholder="Ej: prioriza casos de categoría Finanzas"
                )
                nombre_salida = gr.Textbox(label="Nombre del archivo de salida (opcional)")

                with gr.Row():
                    btn_diagnostico = gr.Button("🔍 Diagnóstico rápido (1 registro)")
                    btn_generar = gr.Button("🚀 Generar dataset", variant="primary")
                    btn_borrar_generar = gr.Button("🗑️ Borrar todo (reset)", variant="stop")

                with gr.Accordion("📋 Resultado del diagnóstico rápido", open=True):
                    diagnostico_output = gr.Textbox(label="Respuesta cruda + resumen", lines=10)
                    diagnostico_errores_output = gr.Textbox(label="Errores del diagnóstico", lines=5)

                with gr.Accordion("📊 Resultado de la generación completa", open=True):
                    preview_output = gr.Textbox(label="Preview (primeros 5 registros)", lines=15)
                    ruta_output = gr.Textbox(label="Archivo guardado en", interactive=False)
                    estado_output = gr.Textbox(label="Estado", interactive=False)
                    errores_output = gr.Textbox(label="Detalle de errores de validación", lines=8)

            # --- Pestaña: Resultados generados ---
            with gr.TabItem("📁 Resultados generados"):
                gr.Markdown("Consulta, limpia la vista o elimina ficheros ya generados en Outbound files.")
                resultados_dropdown = gr.Dropdown(choices=listar_archivos_salida(), label="Archivos generados")
                btn_refrescar_resultados = gr.Button("🔄 Refrescar", size="sm")
                resultados_texto = gr.Textbox(label="Contenido del archivo", lines=15)
                with gr.Row():
                    btn_cargar_resultado = gr.Button("📂 Cargar seleccionado")
                    btn_limpiar_vista_resultado = gr.Button("🧹 Limpiar vista")
                    btn_eliminar_resultado = gr.Button("🗑️ Eliminar archivo", variant="stop")
                resultados_estado = gr.Textbox(label="Estado", interactive=False)

        gr.Markdown(f"---\n<small>José María Rivas | Práctica LLM Engineering · v5 · {datetime.now().strftime('%d/%m/%Y')}</small>")

    # ══════════════ Eventos: arranque ══════════════
    app.load(fn=cargar_sistema, outputs=[log_carga, btn_continuar])

    btn_continuar.click(
        fn=lambda: (gr.update(visible=False), gr.update(visible=True)),
        outputs=[grupo_carga, grupo_principal]
    )

    # ══════════════ Eventos: pestaña Prompt ══════════════
    def _refrescar_prompt():
        archivos = listar_archivos_prompt()
        primero = archivos[0] if archivos else None
        return gr.update(choices=archivos, value=primero)

    btn_refrescar_prompt.click(fn=_refrescar_prompt, outputs=prompt_dropdown)
    btn_cargar_prompt.click(fn=lambda nombre: cargar_prompt(nombre) if nombre else "", inputs=prompt_dropdown, outputs=prompt_texto)
    def _mejorar_prompt_ui(texto, modelo_ia):
        if modelo_ia == "llama" and llama_model is None:
            yield gr.update(), "⏳ Cargando modelo Llama local, puede tardar varios minutos la primera vez..."
        try:
            resultado = enriquecer_prompt_con_ia(texto, modelo=modelo_ia)
            yield resultado, "✅ Completado"
        except Exception as e:
            yield gr.update(), f"❌ Error: {str(e)}"

    btn_mejorar_prompt.click(fn=_mejorar_prompt_ui, inputs=[prompt_texto, motor_ia_prompt], outputs=[prompt_texto, prompt_estado])

    def _usar_prompt(nombre, prompt_activo_actual):
        if not nombre:
            return prompt_activo_actual, "❌ Selecciona un prompt del desplegable", gr.update(), gr.update()
        contenido = cargar_prompt(nombre)
        return contenido, f"✅ Prompt activo: {nombre}", nombre, contenido

    btn_usar_prompt.click(
        fn=_usar_prompt,
        inputs=[prompt_dropdown, prompt_activo],
        outputs=[prompt_activo, prompt_estado, prompt_activo_display, prompt_texto]
    )

    def _guardar_prompt_y_usar(texto, nombre):
        nombre_final = nombre.strip() if nombre and nombre.strip() else generar_nombre_archivo("prompt", "json")
        guardar_prompt(texto, nombre_final)
        return texto, f"✅ Prompt activo: {nombre_final}", nombre_final

    btn_guardar_prompt.click(
        fn=_guardar_prompt_y_usar,
        inputs=[prompt_texto, prompt_nombre_guardar],
        outputs=[prompt_activo, prompt_estado, prompt_activo_display]
    )

    def _borrar_prompt():
        return gr.update(value=None), "", "", ""

    btn_borrar_prompt.click(
        fn=_borrar_prompt,
        outputs=[prompt_dropdown, prompt_texto, prompt_nombre_guardar, prompt_estado]
    )

    def _eliminar_prompt_ui(nombre):
        if not nombre:
            return gr.update(), "❌ Selecciona un prompt del desplegable"
        eliminado = eliminar_archivo(PROMPT_PATH, nombre)
        estado = f"🗑️ Prompt eliminado: {nombre}" if eliminado else f"❌ No se encontró: {nombre}"
        return gr.update(choices=listar_archivos_prompt(), value=None), estado

    btn_eliminar_prompt.click(fn=_eliminar_prompt_ui, inputs=prompt_dropdown, outputs=[prompt_dropdown, prompt_estado])

    # ══════════════ Eventos: pestaña Esquema ══════════════
    def _refrescar_esquema():
        archivos = listar_archivos_esquema()
        primero = archivos[0] if archivos else None
        return gr.update(choices=archivos, value=primero)

    btn_refrescar_esquema.click(fn=_refrescar_esquema, outputs=esquema_dropdown)

    def _cargar_esquema_ui(nombre):
        if not nombre:
            return ""
        esquema = cargar_esquema(nombre)
        return json.dumps(esquema, indent=2, ensure_ascii=False)

    btn_cargar_esquema.click(fn=_cargar_esquema_ui, inputs=esquema_dropdown, outputs=esquema_texto)

    def _validar_esquema_ui(texto, modelo_ia):
        if modelo_ia == "llama" and llama_model is None:
            yield gr.update(), "⏳ Cargando modelo Llama local, puede tardar varios minutos la primera vez..."
        try:
            esquema = validar_formatear_esquema_con_ia(texto, modelo=modelo_ia)
            yield json.dumps(esquema, indent=2, ensure_ascii=False), "✅ Completado"
        except Exception as e:
            yield gr.update(), f"❌ Error: {str(e)}"

    btn_validar_esquema.click(fn=_validar_esquema_ui, inputs=[esquema_texto, motor_ia_esquema], outputs=[esquema_texto, esquema_estado])

    def _usar_esquema(nombre, esquema_activo_actual):
        if not nombre:
            return esquema_activo_actual, "❌ Selecciona un esquema del desplegable", gr.update(), gr.update(), gr.update()
        esquema = cargar_esquema(nombre)
        campos_txt = ", ".join(c["nombre"] for c in esquema["campos"])
        texto_json = json.dumps(esquema, indent=2, ensure_ascii=False)
        return esquema, f"✅ Esquema activo: {nombre}", nombre, campos_txt, texto_json

    btn_usar_esquema.click(
        fn=_usar_esquema,
        inputs=[esquema_dropdown, esquema_activo],
        outputs=[esquema_activo, esquema_estado, esquema_activo_display, esquema_campos_display, esquema_texto]
    )

    def _guardar_esquema_y_usar(texto, nombre):
        esquema = json.loads(texto)
        nombre_final = nombre.strip() if nombre and nombre.strip() else generar_nombre_archivo("esquema", "json")
        guardar_esquema(esquema, nombre_final)
        campos_txt = ", ".join(c["nombre"] for c in esquema["campos"])
        return esquema, f"✅ Esquema activo: {nombre_final}", nombre_final, campos_txt

    btn_guardar_esquema.click(
        fn=_guardar_esquema_y_usar,
        inputs=[esquema_texto, esquema_nombre_guardar],
        outputs=[esquema_activo, esquema_estado, esquema_activo_display, esquema_campos_display]
    )

    def _borrar_esquema():
        return gr.update(value=None), "", "", ""

    btn_borrar_esquema.click(
        fn=_borrar_esquema,
        outputs=[esquema_dropdown, esquema_texto, esquema_nombre_guardar, esquema_estado]
    )

    def _eliminar_esquema_ui(nombre):
        if not nombre:
            return gr.update(), "❌ Selecciona un esquema del desplegable"
        eliminado = eliminar_archivo(SCHEMA_PATH, nombre)
        estado = f"🗑️ Esquema eliminado: {nombre}" if eliminado else f"❌ No se encontró: {nombre}"
        return gr.update(choices=listar_archivos_esquema(), value=None), estado

    btn_eliminar_esquema.click(fn=_eliminar_esquema_ui, inputs=esquema_dropdown, outputs=[esquema_dropdown, esquema_estado])

    # ══════════════ Eventos: pestaña Semilla ══════════════
    def _refrescar_semilla():
        archivos = listar_archivos_semilla()
        primero = archivos[0] if archivos else None
        return gr.update(choices=archivos, value=primero)

    btn_refrescar_semilla.click(fn=_refrescar_semilla, outputs=semilla_dropdown)

    def _cargar_semilla_ui(nombre):
        if not nombre:
            return ""
        datos = cargar_dataset_semilla(nombre)
        return json.dumps(datos, indent=2, ensure_ascii=False)

    btn_cargar_semilla.click(fn=_cargar_semilla_ui, inputs=semilla_dropdown, outputs=semilla_texto)

    def _formatear_semilla_ui(texto, esquema, modelo_ia):
        if modelo_ia == "llama" and llama_model is None:
            yield gr.update(), "⏳ Cargando modelo Llama local, puede tardar varios minutos la primera vez..."
        try:
            datos = formatear_semilla_con_ia(texto, esquema, modelo=modelo_ia)
            yield json.dumps(datos, indent=2, ensure_ascii=False), "✅ Completado"
        except Exception as e:
            yield gr.update(), f"❌ Error: {str(e)}"

    btn_formatear_semilla.click(
        fn=_formatear_semilla_ui,
        inputs=[semilla_texto, esquema_activo, motor_ia_semilla],
        outputs=[semilla_texto, semilla_estado]
    )

    def _usar_semilla(nombre, semilla_activa_actual):
        if not nombre:
            return semilla_activa_actual, "❌ Selecciona una semilla del desplegable", gr.update(), gr.update()
        datos = cargar_dataset_semilla(nombre)
        texto_json = json.dumps(datos, indent=2, ensure_ascii=False)
        return nombre, f"✅ Semilla activa: {nombre}", nombre, texto_json

    btn_usar_semilla.click(
        fn=_usar_semilla,
        inputs=[semilla_dropdown, semilla_activa],
        outputs=[semilla_activa, semilla_estado, semilla_activa_display, semilla_texto]
    )

    def _guardar_semilla_y_usar(texto, nombre):
        datos = json.loads(texto)
        nombre_final = nombre.strip() if nombre and nombre.strip() else generar_nombre_archivo("semilla", "json")
        if not nombre_final.lower().endswith(".json"):
            nombre_final += ".json"
        guardar_dataset_semilla(datos, nombre_final)
        return nombre_final, f"✅ Semilla activa: {nombre_final}", nombre_final

    btn_guardar_semilla.click(
        fn=_guardar_semilla_y_usar,
        inputs=[semilla_texto, semilla_nombre_guardar],
        outputs=[semilla_activa, semilla_estado, semilla_activa_display]
    )

    def _borrar_semilla():
        return gr.update(value=None), "", "", ""

    btn_borrar_semilla.click(
        fn=_borrar_semilla,
        outputs=[semilla_dropdown, semilla_texto, semilla_nombre_guardar, semilla_estado]
    )

    def _eliminar_semilla_ui(nombre):
        if not nombre:
            return gr.update(), "❌ Selecciona una semilla del desplegable"
        eliminado = eliminar_archivo(SEED_PATH, nombre)
        estado = f"🗑️ Semilla eliminada: {nombre}" if eliminado else f"❌ No se encontró: {nombre}"
        return gr.update(choices=listar_archivos_semilla(), value=None), estado

    btn_eliminar_semilla.click(fn=_eliminar_semilla_ui, inputs=semilla_dropdown, outputs=[semilla_dropdown, semilla_estado])

    # ══════════════ Eventos: pestaña Generar ══════════════
    def _diagnostico_rapido_ui(semilla, esquema, prompt, modelo_sel, instrucciones):
        if not semilla or not esquema or not prompt:
            yield "", "❌ Faltan ficheros activos: revisa las pestañas Prompt, Esquema y Semilla."
            return
        if modelo_sel == "llama" and llama_model is None:
            yield gr.update(), "⏳ Cargando modelo Llama local, puede tardar varios minutos la primera vez..."
        try:
            seed_dataset = cargar_dataset_semilla(semilla)
            system_prompt_reforzado = reforzar_prompt_con_esquema(prompt, esquema)
            siguiente_id = calcular_siguiente_id(seed_dataset, esquema.get("campo_id"))

            texto_crudo = generar_registros_raw(
                system_prompt_reforzado, seed_dataset, 1, siguiente_id, esquema,
                modelo=modelo_sel, instrucciones_adicionales=instrucciones
            )
            modelo_validacion = construir_modelo_validacion(esquema)
            validos, errores = parsear_y_validar(texto_crudo, modelo_validacion)

            resumen = f"=== RESPUESTA CRUDA DEL MODELO ===\n{texto_crudo}\n\n=== Válidos: {len(validos)} | Errores: {len(errores)} ==="
            detalle_errores = "\n\n".join(errores) if errores else "Sin errores"
            yield resumen, detalle_errores
        except Exception as e:
            yield "", f"❌ Error: {str(e)}"

    btn_diagnostico.click(
        fn=_diagnostico_rapido_ui,
        inputs=[semilla_activa, esquema_activo, prompt_activo, modelo, instrucciones_adicionales],
        outputs=[diagnostico_output, diagnostico_errores_output]
    )

    def _generar_ui(semilla, esquema, prompt, num_ejemplos, modelo_sel, formato_salida, instrucciones, nombre_salida):
        if not semilla or not esquema or not prompt:
            yield "", "", "❌ Faltan ficheros activos: revisa las pestañas Prompt, Esquema y Semilla.", ""
            return
        if modelo_sel == "llama" and llama_model is None:
            yield gr.update(), gr.update(), "⏳ Cargando modelo Llama local, puede tardar varios minutos la primera vez...", gr.update()
        try:
            registros, ruta, errores = generar_dataset_sintetico(
                num_ejemplos=int(num_ejemplos),
                modelo=modelo_sel,
                formato_salida=formato_salida,
                nombre_semilla=semilla,
                esquema=esquema,
                system_prompt=prompt,
                nombre_salida=nombre_salida if nombre_salida.strip() != "" else None,
                instrucciones_adicionales=instrucciones
            )
            preview = json.dumps(registros[:5], indent=2, ensure_ascii=False)
            estado = f"✅ {len(registros)} registros generados, {len(errores)} con error.\nGuardado en:\n{ruta}"
            detalle_errores = "\n\n".join(errores) if errores else "Sin errores"
            yield preview, ruta, estado, detalle_errores
        except Exception as e:
            yield "", "", f"❌ Error: {str(e)}", ""

    btn_generar.click(
        fn=_generar_ui,
        inputs=[semilla_activa, esquema_activo, prompt_activo, num_ejemplos, modelo,
                formato_salida, instrucciones_adicionales, nombre_salida],
        outputs=[preview_output, ruta_output, estado_output, errores_output]
    )

    def _borrar_generar():
        return (
            None, None, None,
            "", "", "", "",
            10, "claude", "json", "", "",
            "", "",
            "", "", "", ""
        )

    btn_borrar_generar.click(
        fn=_borrar_generar,
        outputs=[
            semilla_activa, esquema_activo, prompt_activo,
            semilla_activa_display, esquema_activo_display, prompt_activo_display, esquema_campos_display,
            num_ejemplos, modelo, formato_salida, instrucciones_adicionales, nombre_salida,
            diagnostico_output, diagnostico_errores_output,
            preview_output, ruta_output, estado_output, errores_output
        ]
    )

    # ══════════════ Eventos: pestaña Resultados generados ══════════════
    def _refrescar_resultados():
        archivos = listar_archivos_salida()
        primero = archivos[0] if archivos else None
        return gr.update(choices=archivos, value=primero)

    btn_refrescar_resultados.click(fn=_refrescar_resultados, outputs=resultados_dropdown)

    def _cargar_resultado_ui(nombre):
        if not nombre:
            return "", "❌ Selecciona un archivo del desplegable"
        contenido = cargar_archivo_salida(nombre)
        return contenido, f"✅ Mostrando: {nombre}"

    btn_cargar_resultado.click(fn=_cargar_resultado_ui, inputs=resultados_dropdown, outputs=[resultados_texto, resultados_estado])

    def _limpiar_vista_resultado():
        return "", ""

    btn_limpiar_vista_resultado.click(fn=_limpiar_vista_resultado, outputs=[resultados_texto, resultados_estado])

    def _eliminar_resultado_ui(nombre):
        if not nombre:
            return "", gr.update(), "❌ Selecciona un archivo del desplegable"
        eliminado = eliminar_archivo(OUTPUT_PATH, nombre)
        estado = f"🗑️ Archivo eliminado: {nombre}" if eliminado else f"❌ No se encontró: {nombre}"
        return "", gr.update(choices=listar_archivos_salida(), value=None), estado

    btn_eliminar_resultado.click(
        fn=_eliminar_resultado_ui,
        inputs=resultados_dropdown,
        outputs=[resultados_texto, resultados_dropdown, resultados_estado]
    )


In [49]:
app.launch(share=True) #, debug=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc149cc48239312c38.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
